# YOLO-OBB Review Crops

This notebook is separate from the main `ParkingYoloOBB_pipeline.ipynb` on purpose.

The main pipeline should stay clean and repeatable:

`input imagery -> train/predict YOLO-OBB -> export georeferenced detections`

This notebook is for preparation of review material:

`YOLO-OBB predictions -> straighten each detected parking bay crop -> sort by confidence -> prepare images for website / municipality review`

The output is not a model-training product. It is a practical communication and review layer. Predictions whose OBB corners fall outside the tile, or whose crop cannot keep enough context, are skipped. Saved crops are normalized onto a fixed `320 x 480` canvas for website display.

## Required input data

You need two input folders.

1. **Prediction images**
   - These are the same images that YOLO used during prediction.
   - They can be `.png`, `.jpg`, `.jpeg`, `.tif`, or `.tiff`.
   - The image filename stem must match the prediction label filename stem.
   - Example: `tile_001.png` matches `tile_001.txt`.

2. **YOLO-OBB prediction labels**
   - These are the `.txt` files from the YOLO-OBB prediction run.
   - They are usually created when prediction is run with `save_txt=True` and `save_conf=True`.
   - Expected line format:

```text
class_id x1 y1 x2 y2 x3 y3 x4 y4 confidence
```

The corner coordinates should be normalized YOLO coordinates between `0` and `1`.

If the confidence value is missing, the notebook can still crop the detections, but sorting by confidence will be less useful.

## Recommended Colab folder structure

This matches `ParkingYoloOBB_pipeline.ipynb`. Upload the project folder to Google Drive under `MyDrive/ParkingSpaceDetection`.

The important link is simple: image `101.png` must match prediction label `101.txt`.

```text
/content/gdrive/MyDrive/ParkingSpaceDetection/
  scripts/

  GeoreferenceTest/
    ParkingYoloOBB_pipeline.ipynb
    ParkingYoloOBB_review_crops.ipynb

    202606121032215360267/
      png/                            # all converted PNG tiles

      dataset/
        images/val/                   # default source images used for prediction
          101.png
          104.png
        labels/val/                   # ground truth labels, not used for cropping predictions

      runs/obb/
        predict/                      # first prediction run, if it exists
          labels/
            101.txt
            104.txt

        predict-2/                    # Colab/Ultralytics often creates this on rerun
          labels/
            101.txt
            104.txt

      review_crops/                   # created by this notebook
        all/                          # every crop, sorted by confidence in filename
        high_confidence/              # confidence >= 0.80
        medium_confidence/            # confidence >= 0.50 and < 0.80
        low_confidence/               # confidence < 0.50
        unknown_confidence/           # labels without confidence column
        manifest.csv                  # table for website/review use
```

The notebook automatically uses the newest `runs/obb/predict*` folder. If you predicted on all PNGs instead of validation images, set `IMAGE_DIR = PNG_DIR` in the next cell.

In [ ]:
from pathlib import Path

# Same Google Drive base path as ParkingYoloOBB_pipeline.ipynb.
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ModuleNotFoundError:
    pass

BASE = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')
RUN_DIR = BASE / 'GeoreferenceTest/202606121032215360267'

PNG_DIR = RUN_DIR / 'png'
DATASET_DIR = RUN_DIR / 'dataset'

def newest_predict_label_dir(run_dir: Path):
    predict_root = run_dir / 'runs/obb'
    candidates = [p for p in predict_root.glob('predict*') if (p / 'labels').exists()]
    if not candidates:
        return predict_root / 'predict/labels'
    return max(candidates, key=lambda p: p.stat().st_mtime) / 'labels'

# This should match the folder used as PREDICT_SOURCE in the pipeline notebook.
# The pipeline currently predicts on validation images.
# If you predict on all PNG tiles instead, change this to: IMAGE_DIR = PNG_DIR
IMAGE_DIR = DATASET_DIR / 'images/val'
LABEL_DIR = newest_predict_label_dir(RUN_DIR)
OUTPUT_DIR = RUN_DIR / 'review_crops'

# Crop settings
# The crop is rectified: the predicted OBB is warped into a straight rectangle.
# Padding is attempted, then reduced automatically if it would leave the tile.
PADDING_PX = 24
PADDING_RATIO = 0.30
MIN_PADDING_FRACTION = 0.50
OUTPUT_CANVAS_SIZE = (320, 480)
CANVAS_BACKGROUND = (244, 241, 234)
DRAW_OBB_OUTLINE = True
SKIP_OBB_OUTSIDE_TILE = True

# Confidence bands for easier website/review preparation.
HIGH_CONFIDENCE = 0.80
MEDIUM_CONFIDENCE = 0.50

print('Base folder:', BASE, BASE.exists())
print('Run folder:', RUN_DIR, RUN_DIR.exists())
print('PNG folder:', PNG_DIR, PNG_DIR.exists())
print('Dataset folder:', DATASET_DIR, DATASET_DIR.exists())
print('Image folder:', IMAGE_DIR, IMAGE_DIR.exists())
print('Label folder:', LABEL_DIR, LABEL_DIR.exists())
print('Output folder:', OUTPUT_DIR)

In [ ]:
from PIL import Image, ImageDraw, ImageOps
import csv
import math
import shutil
from IPython.display import display

IMAGE_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.tif', '.tiff']

def find_image_by_stem(image_dir: Path, stem: str):
    for ext in IMAGE_EXTENSIONS:
        candidate = image_dir / f'{stem}{ext}'
        if candidate.exists():
            return candidate
    return None

def parse_obb_label_line(line: str, source_label: Path, line_number: int):
    parts = line.strip().split()
    if not parts:
        return None
    if len(parts) not in (9, 10):
        raise ValueError(
            f'Unexpected OBB label format in {source_label}, line {line_number}. '
            f'Expected 9 or 10 values, got {len(parts)}: {line!r}'
        )

    class_id = int(float(parts[0]))
    coords = [float(v) for v in parts[1:9]]
    confidence = float(parts[9]) if len(parts) == 10 else None

    return {
        'class_id': class_id,
        'coords_norm': coords,
        'confidence': confidence,
        'source_label': source_label,
        'line_number': line_number,
    }

def normalized_obb_to_pixel_points(coords_norm, image_width, image_height):
    points = []
    for i in range(0, 8, 2):
        x = coords_norm[i] * image_width
        y = coords_norm[i + 1] * image_height
        points.append((x, y))
    return points

def confidence_band(confidence):
    if confidence is None or math.isnan(confidence):
        return 'unknown_confidence'
    if confidence >= HIGH_CONFIDENCE:
        return 'high_confidence'
    if confidence >= MEDIUM_CONFIDENCE:
        return 'medium_confidence'
    return 'low_confidence'

def safe_confidence_for_sort(confidence):
    return -1 if confidence is None or math.isnan(confidence) else confidence

def clamp(value, lower, upper):
    return max(lower, min(upper, value))

def explain_drive_error(error, image_file):
    if isinstance(error, OSError) and getattr(error, 'errno', None) == 107:
        raise RuntimeError(
            'Google Drive disconnected while reading an image. '
            'Run the path/setup cell again to remount Drive, then rerun this cell. '
            f'Failed image: {image_file}'
        ) from error
    raise error

def point_inside_image(point, image_width, image_height):
    x, y = point
    return 0 <= x <= image_width and 0 <= y <= image_height

def points_inside_image(points, image_width, image_height):
    return all(point_inside_image(point, image_width, image_height) for point in points)

def distance(a, b):
    return math.hypot(b[0] - a[0], b[1] - a[1])

def unit_vector(a, b):
    length = distance(a, b)
    if length == 0:
        return None
    return ((b[0] - a[0]) / length, (b[1] - a[1]) / length)

def padded_rectified_crop_geometry(points_px, image_width, image_height):
    if SKIP_OBB_OUTSIDE_TILE and not points_inside_image(points_px, image_width, image_height):
        return None, 'outside_tile'

    p0, p1, p2, p3 = points_px
    u = unit_vector(p0, p1)
    v = unit_vector(p0, p3)
    if u is None or v is None:
        return None, 'invalid_obb'

    obb_width = (distance(p0, p1) + distance(p3, p2)) / 2
    obb_height = (distance(p1, p2) + distance(p0, p3)) / 2
    if obb_width < 2 or obb_height < 2:
        return None, 'invalid_obb'

    desired_pad_x = max(PADDING_PX, obb_width * PADDING_RATIO)
    desired_pad_y = max(PADDING_PX, obb_height * PADDING_RATIO)

    def make_quad(scale):
        pad_x = desired_pad_x * scale
        pad_y = desired_pad_y * scale
        q0 = (p0[0] - pad_x * u[0] - pad_y * v[0], p0[1] - pad_x * u[1] - pad_y * v[1])
        q1 = (p1[0] + pad_x * u[0] - pad_y * v[0], p1[1] + pad_x * u[1] - pad_y * v[1])
        q2 = (p2[0] + pad_x * u[0] + pad_y * v[0], p2[1] + pad_x * u[1] + pad_y * v[1])
        q3 = (p3[0] - pad_x * u[0] + pad_y * v[0], p3[1] - pad_x * u[1] + pad_y * v[1])
        return [q0, q1, q2, q3], pad_x, pad_y

    quad, pad_x, pad_y = make_quad(1.0)
    if not points_inside_image(quad, image_width, image_height):
        lo, hi = 0.0, 1.0
        for _ in range(24):
            mid = (lo + hi) / 2
            test_quad, _, _ = make_quad(mid)
            if points_inside_image(test_quad, image_width, image_height):
                lo = mid
            else:
                hi = mid
        quad, pad_x, pad_y = make_quad(lo)

    padding_fraction = min(pad_x / desired_pad_x, pad_y / desired_pad_y)
    if padding_fraction < MIN_PADDING_FRACTION:
        return None, 'low_context'

    output_width = max(1, int(round(obb_width + 2 * pad_x)))
    output_height = max(1, int(round(obb_height + 2 * pad_y)))
    xs = [p[0] for p in quad]
    ys = [p[1] for p in quad]

    return {
        'source_quad': quad,
        'output_size': (output_width, output_height),
        'padding_x_px': pad_x,
        'padding_y_px': pad_y,
        'padding_fraction': padding_fraction,
        'crop_box': (
            int(math.floor(min(xs))),
            int(math.floor(min(ys))),
            int(math.ceil(max(xs))),
            int(math.ceil(max(ys))),
        ),
    }, None

def rectified_obb_crop(image, geometry):
    q0, q1, q2, q3 = geometry['source_quad']
    # PIL QUAD order is upper-left, lower-left, lower-right, upper-right.
    quad_for_pil = [q0[0], q0[1], q3[0], q3[1], q2[0], q2[1], q1[0], q1[1]]
    return image.transform(
        geometry['output_size'],
        Image.Transform.QUAD,
        quad_for_pil,
        resample=Image.Resampling.BICUBIC,
    )

def force_portrait_crop(crop):
    if crop.width > crop.height:
        return crop.rotate(90, expand=True), True
    return crop, False

def normalize_to_canvas(crop):
    crop, was_rotated = force_portrait_crop(crop)
    canvas_width, canvas_height = OUTPUT_CANVAS_SIZE
    fitted = ImageOps.contain(crop, OUTPUT_CANVAS_SIZE, method=Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', OUTPUT_CANVAS_SIZE, CANVAS_BACKGROUND)
    x = (canvas_width - fitted.width) // 2
    y = (canvas_height - fitted.height) // 2
    canvas.paste(fitted, (x, y))
    return canvas, was_rotated

In [ ]:
# Collect all detections first, so they can be sorted globally by confidence.

if not IMAGE_DIR.exists():
    raise FileNotFoundError(f'IMAGE_DIR does not exist: {IMAGE_DIR}')

if not LABEL_DIR.exists():
    raise FileNotFoundError(f'LABEL_DIR does not exist: {LABEL_DIR}')

detections = []
missing_images = []
skipped_outside_tile = 0
skipped_low_context = 0
skipped_invalid_obb = 0

for label_file in sorted(LABEL_DIR.glob('*.txt')):
    image_file = find_image_by_stem(IMAGE_DIR, label_file.stem)
    if image_file is None:
        missing_images.append(label_file.name)
        continue

    try:
        with Image.open(image_file) as img:
            width, height = img.size
    except OSError as error:
        explain_drive_error(error, image_file)

    with label_file.open('r', encoding='utf-8') as f:
        for line_number, line in enumerate(f, start=1):
            parsed = parse_obb_label_line(line, label_file, line_number)
            if parsed is None:
                continue

            points_px = normalized_obb_to_pixel_points(parsed['coords_norm'], width, height)
            geometry, skip_reason = padded_rectified_crop_geometry(points_px, width, height)
            if geometry is None:
                if skip_reason == 'outside_tile':
                    skipped_outside_tile += 1
                elif skip_reason == 'low_context':
                    skipped_low_context += 1
                else:
                    skipped_invalid_obb += 1
                continue

            parsed.update({
                'source_image': image_file,
                'image_width': width,
                'image_height': height,
                'points_px': points_px,
                'crop_box': geometry['crop_box'],
                'rectified_geometry': geometry,
                'confidence_band': confidence_band(parsed['confidence']),
            })
            detections.append(parsed)

detections = sorted(detections, key=lambda d: safe_confidence_for_sort(d['confidence']), reverse=True)

print(f'Found {len(detections)} detections.')
print(f'Skipped outside-tile OBBs: {skipped_outside_tile}')
print(f'Skipped low-context OBBs: {skipped_low_context}')
print(f'Skipped invalid OBBs: {skipped_invalid_obb}')
if missing_images:
    print(f'Warning: {len(missing_images)} label files had no matching image file.')
    print(missing_images[:10])

In [ ]:
# Write crops, confidence-band folders, and manifest.csv.

all_dir = OUTPUT_DIR / 'all'
band_dirs = {
    'high_confidence': OUTPUT_DIR / 'high_confidence',
    'medium_confidence': OUTPUT_DIR / 'medium_confidence',
    'low_confidence': OUTPUT_DIR / 'low_confidence',
    'unknown_confidence': OUTPUT_DIR / 'unknown_confidence',
}

# Clear old crop files so skipped edge detections do not leave stale images behind.
for folder in [all_dir, *band_dirs.values()]:
    if folder.exists():
        shutil.rmtree(folder)

all_dir.mkdir(parents=True, exist_ok=True)
for folder in band_dirs.values():
    folder.mkdir(parents=True, exist_ok=True)

manifest_path = OUTPUT_DIR / 'manifest.csv'

fieldnames = [
    'rank', 'crop_file', 'confidence_band', 'source_image', 'source_label', 'line_number',
    'class_id', 'confidence', 'image_width', 'image_height',
    'crop_xmin', 'crop_ymin', 'crop_xmax', 'crop_ymax',
    'rectified_crop_width', 'rectified_crop_height', 'padding_x_px', 'padding_y_px',
    'padding_fraction', 'canvas_width', 'canvas_height', 'rotated_to_portrait',
    'x1_norm', 'y1_norm', 'x2_norm', 'y2_norm', 'x3_norm', 'y3_norm', 'x4_norm', 'y4_norm',
    'x1_px', 'y1_px', 'x2_px', 'y2_px', 'x3_px', 'y3_px', 'x4_px', 'y4_px',
]

with manifest_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for rank, det in enumerate(detections, start=1):
        conf = det['confidence']
        conf_text = 'no_conf' if conf is None else f'{conf:.3f}'
        source_stem = det['source_image'].stem
        crop_name = f'{rank:05d}_conf_{conf_text}_{source_stem}_line_{det["line_number"]}.png'

        crop_path = all_dir / crop_name
        band_path = band_dirs[det['confidence_band']] / crop_name

        try:
            with Image.open(det['source_image']) as img:
                img = img.convert('RGB')
                geometry = det['rectified_geometry']
                crop = rectified_obb_crop(img, geometry)

                if DRAW_OBB_OUTLINE:
                    crop_w, crop_h = geometry['output_size']
                    pad_x = geometry['padding_x_px']
                    pad_y = geometry['padding_y_px']
                    draw = ImageDraw.Draw(crop)
                    draw.rectangle(
                        (pad_x, pad_y, crop_w - pad_x, crop_h - pad_y),
                        outline=(255, 0, 0),
                        width=3,
                    )

                crop, rotated_to_portrait = normalize_to_canvas(crop)
                crop.save(crop_path)
        except OSError as error:
            explain_drive_error(error, det['source_image'])

        shutil.copy2(crop_path, band_path)

        coords_norm = det['coords_norm']
        points_px = det['points_px']
        crop_xmin, crop_ymin, crop_xmax, crop_ymax = det['crop_box']
        geometry = det['rectified_geometry']
        crop_width, crop_height = geometry['output_size']

        writer.writerow({
            'rank': rank,
            'crop_file': str(crop_path),
            'confidence_band': det['confidence_band'],
            'source_image': str(det['source_image']),
            'source_label': str(det['source_label']),
            'line_number': det['line_number'],
            'class_id': det['class_id'],
            'confidence': '' if conf is None else conf,
            'image_width': det['image_width'],
            'image_height': det['image_height'],
            'crop_xmin': crop_xmin,
            'crop_ymin': crop_ymin,
            'crop_xmax': crop_xmax,
            'crop_ymax': crop_ymax,
            'rectified_crop_width': crop_width,
            'rectified_crop_height': crop_height,
            'padding_x_px': geometry['padding_x_px'],
            'padding_y_px': geometry['padding_y_px'],
            'padding_fraction': geometry['padding_fraction'],
            'canvas_width': OUTPUT_CANVAS_SIZE[0],
            'canvas_height': OUTPUT_CANVAS_SIZE[1],
            'rotated_to_portrait': rotated_to_portrait,
            'x1_norm': coords_norm[0], 'y1_norm': coords_norm[1],
            'x2_norm': coords_norm[2], 'y2_norm': coords_norm[3],
            'x3_norm': coords_norm[4], 'y3_norm': coords_norm[5],
            'x4_norm': coords_norm[6], 'y4_norm': coords_norm[7],
            'x1_px': points_px[0][0], 'y1_px': points_px[0][1],
            'x2_px': points_px[1][0], 'y2_px': points_px[1][1],
            'x3_px': points_px[2][0], 'y3_px': points_px[2][1],
            'x4_px': points_px[3][0], 'y4_px': points_px[3][1],
        })

print(f'Wrote crops to: {OUTPUT_DIR}')
print(f'Wrote manifest to: {manifest_path}')

In [ ]:
# Quick visual check: show the top detections by confidence.

preview_files = sorted((OUTPUT_DIR / 'all').glob('*.png'))[:12]

print(f'Showing {len(preview_files)} preview crops.')
for crop_file in preview_files:
    print(crop_file.name)
    display(Image.open(crop_file))

## How to use the output

- `review_crops/all/` contains every straightened cropped detection on a fixed `320 x 480` canvas, sorted by confidence in the filename.
- `review_crops/high_confidence/` contains detections that are probably reliable.
- `review_crops/medium_confidence/` contains detections that are useful for human checking.
- `review_crops/low_confidence/` contains detections that are likely noisy but still useful for error analysis.
- `review_crops/manifest.csv` links every crop back to its source image, label file, line number, confidence, OBB coordinates, applied padding, padding fraction, rectified crop size, and canvas size.
- Detections with OBB corners outside the tile are skipped because a clean padded crop is impossible there.
- Detections with less than `50%` of the requested context padding are skipped because they look bad in review.

For a simple website, start from `manifest.csv` and display the crop images in confidence order. Later, this can become a human-in-the-loop review page where the municipality marks detections as accepted, rejected, or uncertain.